In [2]:
import anthropic
from dotenv import load_dotenv
import os
import json
from datetime import datetime
load_dotenv()
client = anthropic.Anthropic()
TRANSACTIONS_FILE = "MMA-Dataset.json"
messages=[]


tools=[{"name":"add_fight","description":"Add a MMA fight to the record, with a winner and win method, also add my raiting of the fight",        
        "input_schema":{
        "type": "object",
        "properties": {
            "fighter_one": {"type": "string", "description": "First fighter"},
            "fighter_two": {"type": "string", "description": "Second fighter"},
            "weight_class": {"type": "string", "description": "The weight class of the fight, this possible as follows: atomweight,strawweight, flyweight, bantamweight, featherweight, ligthweight, welterweight, middleweight, light heavyweigth, heavyweight"},
            "rating_from_me": {"type": "number", "description": "Rating of how fun i thought the fight was, rating from 1-10"},
            "winner": {"type": "string", "description": "The fighter who won"},
            "win_method": {"type": "string", "description": "Win method such as knockout, TKO, submission, decison and split decision"},
            "datetoday": {"type": "string", "description": "Date of fight in YYYY-MM-DD format. Leave out to use today's date, remember the year currently is 2026"}
        },
            "required":["fighter_one","fighter_two","weight_class","rating_from_me","winner","win_method"]
        }},
      {"name":"list_fights","description":"Find a list of fights, either by weight class or just everyone",        
        "input_schema":{
        "type": "object",
        "properties": {
            "weight_class": {"type": "string", "description": "The weight class of the fights you look for, if not inputed will return all fights"},
        },
            "required":[]
        }},
      {"name":"delete_fight","description":"Delete fights of a certain id",        
        "input_schema":{
        "type": "object",
        "properties": {
            "fight_id": {"type": "number", "description": "Id of the figth to delete"},
        },
            "required":["fight_id"]
        }}
]

def load_fights():
    if os.path.exists(TRANSACTIONS_FILE):
        with open(TRANSACTIONS_FILE, 'r') as f:
            transactions = json.load(f)
        return transactions
    else:
        return []

def save_fights(transactions):
    with open(TRANSACTIONS_FILE, 'w') as f:
        json.dump(transactions,f,indent=2)

def add_fight(fighter_one, fighter_two, weight_class, rating_from_me,winner,win_method, datetoday=None):
    if datetoday is None:
        datetoday = datetime.now().date().isoformat()
    data = load_fights()
    highest = 0
    for i in data:
        if highest<i["id"]:
            highest=i["id"]
    new_id = highest + 1
    transaction = {
        "id":new_id,
        "fighter_one":fighter_one,
        "fighter_two":fighter_two,
        "weight_class":weight_class,
        "rating_from_me":rating_from_me,
        "winner":winner,
        "win_method":win_method,
        "date":datetoday
    }
    data.append(transaction)
    save_fights(data)
    return transaction

def list_fights(weight_class=None):
    data = load_fights()
    if weight_class is None:
        return data
    returnfights = []
    for i in data:
        if i["weight_class"].lower()==weight_class.lower():
            returnfights.append(i)
    return returnfights

def delete_fight(fight_id):
    data = load_fights()
    deleted = None
    kept = []
    for i in data:
        if i["id"] == fight_id:
            deleted = i
        else:
            kept.append(i)
    if deleted is None:
        return{"Error": "No fight found on this id for deletion"}
    save_fights(kept)
    return deleted

tool_functions = {"add_fight":add_fight,"list_fights":list_fights,"delete_fight":delete_fight}
while True:
    user_text = input()
    if user_text == "exit":
        break
    messages.append({"role":"user","content":user_text})
    message = client.messages.create(
        tools=tools,
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=messages
    )
    messages.append({"role":"assistant","content":message.content})
    while message.stop_reason == "tool_use":
        tool_results = []
        for i in message.content:
            if i.type == "tool_use":
                result = tool_functions[i.name](**i.input)
                tool_results.append({
                    "type":"tool_result",
                    "tool_use_id":i.id,
                    "content":json.dumps(result)
                })
        messages.append({"role":"user","content":tool_results})
        message = client.messages.create(
            tools=tools,
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            messages=messages
        )
        messages.append({"role": "assistant", "content": message.content})
        
    for block in message.content:
        if block.type == "text":
            print(block.text)


 exit
